## Final aggregation of events data

In [ ]:
import os
import re
import numpy as np
import pandas as pd

# ============================================================
# 1) Deduplication helper
# ============================================================

def dedupe_parallel_arrays(df, list_cols):
    """
    Deduplicate primary list column (e.g. SOURCEURL)
    and apply the same index filtering to all columns in list_cols.
    Supports numpy arrays and Python lists.
    """
    primary = list_cols[0]

    def normalize(x):
        if isinstance(x, list):
            return x
        if isinstance(x, np.ndarray):
            return x.tolist()
        return x

    def dedupe_row(row):
        urls = normalize(row[primary])
        if not isinstance(urls, list):
            return row

        seen = set()
        keep_idx = []

        for i, u in enumerate(urls):
            if u not in seen:
                seen.add(u)
                keep_idx.append(i)

        # Apply same filtering to all parallel list columns
        for col in list_cols:
            colval = normalize(row[col])
            if isinstance(colval, list):
                # Only keep indices that are valid for this column's length
                valid_idx = [i for i in keep_idx if i < len(colval)]
                row[col] = [colval[i] for i in valid_idx]

        return row

    return df.apply(dedupe_row, axis=1)

# ============================================================
# 2) Load mapped GDELT dataset
# ============================================================

mapped_path = "../data/gdelt/events/3_mapped/gdelt_mapped.parquet"
if not os.path.exists(mapped_path):
    raise FileNotFoundError(f"Missing mapped GDELT file: {mapped_path}")

df_final = pd.read_parquet(mapped_path)
print("✅ Loaded GDELT mapped dataset:", len(df_final))

# ============================================================
# 3) Fill missing ADMIN0 using lookup
# ============================================================

locations = df_final[['ADMIN0', 'ADMIN1', 'ADMIN2']]
locations = locations[locations['ADMIN0'].notna()].drop_duplicates()

admin_lookup = {
    (row.ADMIN1, row.ADMIN2): row.ADMIN0
    for _, row in locations.iterrows()
}

def fill_admin0(row):
    if pd.isna(row['ADMIN0']):
        return admin_lookup.get((row['ADMIN1'], row['ADMIN2']), None)
    return row['ADMIN0']

df_final['ADMIN0'] = df_final.apply(fill_admin0, axis=1)

# ============================================================
# 4) Deduplicate SOURCEURL + parallel array columns
# ============================================================

list_columns = [
    "SOURCEURL",
    "SQLDATE",
    "EventCode",
    "NumMentions",
    "NumSources",
    "NumArticles"
]

df_final = dedupe_parallel_arrays(df_final, list_columns)
print("✅ Deduplication complete")

# ============================================================
# 5) Load scraped flat dataset & build feature maps
# ============================================================

df_scraped = pd.read_parquet(
    "../data/gdelt/events/5_modelled/events_exploded_with_counts_and_topics.parquet"
)
print("✅ Loaded scraped URLs dataset:", len(df_scraped))

# Build set of **valid scraped URLs**
scraped_urls = set(df_scraped["url"].tolist())

# Add column showing which URLs survived filtering
df_final["valid_SOURCEURL"] = df_final["SOURCEURL"].apply(
    lambda urls: [u for u in urls if u in scraped_urls]
)

# ——— Features to remap ———
feature_cols = [
    'clean_text',
    'NER_admin0', 'NER_admin1', 'NER_admin2',
    'compound_score', 'neg_score', 'neu_score', 'pos_score',
    'fatalities_freq', 'displaced_freq', 'detained_freq',
    'injured_freq', 'sexual_violence_freq', 'torture_freq',
    'economic_shocks_freq', 'agriculture_freq', 'weather_freq',
    'food_insecurity_freq',
    'fatalities_count', 'displaced_count', 'detained_count',
    'injured_count', 'sexual_violence_count', 'torture_count',
    'sentiment.compound', 'sentiment.neg', 'sentiment.neu', 'sentiment.pos',
    'pred_impact_type', 'pred_urgency',
    'pred_resource_food', 'pred_resource_water', 'pred_resource_cash_aid',
    'pred_resource_healthcare', 'pred_resource_shelter',
    'pred_resource_livelihoods', 'pred_resource_education',
    'pred_resource_infrastructure', 'pred_resource_none'
]

# Build dicts for URL → feature
feature_maps = {
    col: df_scraped.set_index("url")[col].to_dict()
    for col in feature_cols
}

def map_list(url_list, mapping):
    if not isinstance(url_list, (list, np.ndarray)):
        return []
    return [mapping[u] for u in url_list if u in mapping]

# ============================================================
# 6) Map all scraped features back into events
# ============================================================

for col, mp in feature_maps.items():
    df_final[col + "_list"] = df_final["SOURCEURL"].apply(
        lambda urls: map_list(urls, mp)
    )

print("✅ All article-level features mapped back into event dataset")

# Only keep event rows where at least one valid clean_text exists
df_final = df_final[
    df_final.clean_text_list.apply(lambda x: isinstance(x, list) and len(x) > 1)
    | df_final.CS_score.isin([1,2,3,4,5])
]

# Cap counts columns to remove unreasonable values
CAPS = {
    "fatalities_count_list": 50,
    "injured_count_list": 100,
    "displaced_count_list": 500,
    "detained_count_list": 50,
    "sexual_violence_count_list": 10,
    "torture_count_list": 10
}

for col, cap in CAPS.items():
    df_final[col] = df_final[col].apply(
        lambda lst: [min(x, cap) for x in lst] if isinstance(lst, list) else lst
    )

print(len(df_final['ADMIN1'].unique()))
print(len(df_final['ADMIN2'].unique()))

# Save final dataset
df_final.to_parquet("../data/gdelt/events/6_final/events_dataset_v2.parquet")
df_final.head()

## Validation and analysis
#### mention the idea of filtering by certain urls

In [ ]:
pd.set_option("display.max_columns", None)
df_final.reset_index(drop=True, inplace=True)
tmp = df_final[df_final['pred_urgency_list'].apply(lambda x: isinstance(x, list) and len(x) > 0)]
tmp.head()

In [ ]:
print(df_final['ADMIN0'].unique())

In [ ]:
max(df_final['fatalities_count_list'].explode())

### Checking article country by period and country

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# -----------------------
# FEWSNET PERIOD LIST
# -----------------------
fewsnet_periods = [
    '201602','201606','201610','201702','201706','201710','201802','201806','201810',
    '201902','201906','201910','202002','202006','202010','202102','202106','202110',
    '202202','202206','202210','202302','202306','202310','202402', '202406'
]

# -----------------------
# 1. BUILD df_heat PIVOT TABLE
# -----------------------
print("Number of unique articles in df_final:")
print(len(df_final.explode("valid_SOURCEURL")['valid_SOURCEURL'].unique()))

df_heat = (
    df_final
        .explode("valid_SOURCEURL")
        .dropna(subset=["valid_SOURCEURL"])
        .groupby(["ADMIN0", "period"])["valid_SOURCEURL"]
        .nunique()
        .unstack(fill_value=0)
        .sort_index()
        .sort_index(axis=1)
)
df_heat.index = df_heat.index.str.title()
# -----------------------
# 2. PLOT HEATMAP
# -----------------------
plt.figure(figsize=(40, 18))
ax = sns.heatmap(
    df_heat,
    cmap="YlOrRd",
    linewidths=0.2,
    linecolor="gray",
    annot=True,
    fmt="d",
    annot_kws={"size": 8},
    cbar_kws={"shrink": 0.5}
)

# -----------------------
# 3. DRAW THICK OUTLINES AROUND FEWSNET COLUMNS
# -----------------------
x_labels = df_heat.columns.tolist()

for p in fewsnet_periods:
    if p in x_labels:
        col_idx = x_labels.index(p)
        
        # Draw a thick rectangle around the column
        ax.add_patch(plt.Rectangle(
            (col_idx, 0),                # (x,y) lower-left corner in data coords
            1,                           # width (1 column)
            df_heat.shape[0],            # height (# of rows)
            fill=False,
            edgecolor='black',            # outline color
            linewidth=3                  # thickness
        ))

# -----------------------
# 4. FORMATTING & LABELS
# -----------------------
ax.set_title(
    "Unique Article Counts by Country and Period (GDELT Events)\nwith FEWSNET Periods Outlined",
    fontsize=26, fontweight='bold', pad=30
)

ax.set_xlabel("Period (YYYYMM)", fontsize=20, labelpad=20)
ax.set_ylabel("Country (ADMIN0)", fontsize=20, labelpad=20)

ax.set_xticklabels(x_labels, fontsize=10, rotation=90)
ax.set_yticklabels(df_heat.index, fontsize=14)

plt.tight_layout()
plt.show()

### Checking max IPC score by period and country

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# -----------------------
# FEWSNET PERIOD LIST
# -----------------------
fewsnet_periods = [
    '201602','201606','201610','201702','201706','201710','201802','201806','201810',
    '201902','201906','201910','202002','202006','202010','202102','202106','202110',
    '202202','202206','202210','202302','202306','202310','202402'
]

# -----------------------
# 1. BUILD df_heat PIVOT TABLE (MAX CS_score)
# -----------------------
assert "CS_score" in df_final.columns, "CS_score column missing from df_final"
assert "ADMIN0" in df_final.columns and "period" in df_final.columns, "Required columns missing"
df_heat = df_final[df_final['CS_score'].isin([1,2,3,4,5])]
df_heat = (
    df_heat
        .explode("valid_SOURCEURL")
        .dropna(subset=["valid_SOURCEURL", "CS_score"])
        .groupby(["ADMIN0", "period"])["CS_score"]
        .mean()
        .unstack(fill_value=np.nan)
        .sort_index()
        .sort_index(axis=1)
)
df_heat.index = df_heat.index.str.title()

# -----------------------
# 2. PLOT HEATMAP
# -----------------------
plt.figure(figsize=(40, 18))
ax = sns.heatmap(
    df_heat,
    cmap="YlOrRd",
    linewidths=0.2,
    linecolor="gray",
    annot=True,
    fmt=".1f",  # IPC phases are integers
    annot_kws={"size": 8},
    cbar_kws={"shrink": 0.5, "label": "Max IPC Phase (CS_score)"}
)

# -----------------------
# 3. DRAW THICK OUTLINES AROUND FEWSNET COLUMNS
# -----------------------
x_labels = df_heat.columns.tolist()

for p in fewsnet_periods:
    if p in x_labels:
        col_idx = x_labels.index(p)
        ax.add_patch(plt.Rectangle(
            (col_idx, 0),                # (x,y) lower-left corner
            1,                           # width (1 column)
            df_heat.shape[0],            # height (# of rows)
            fill=False,
            edgecolor='black',
            linewidth=3
        ))

# -----------------------
# 4. FORMATTING & LABELS
# -----------------------
ax.set_title(
    "Average IPC Phase (CS_score) by Country and Period\nwith FEWSNET Periods Outlined",
    fontsize=26, fontweight='bold', pad=30
)
ax.set_xlabel("Period (YYYYMM)", fontsize=20, labelpad=20)
ax.set_ylabel("Country (ADMIN0)", fontsize=20, labelpad=20)
ax.set_xticklabels(x_labels, fontsize=10, rotation=90)
ax.set_yticklabels(df_heat.index, fontsize=14)

plt.tight_layout()
plt.show()

In [ ]:
df_heat = df_final[df_final['CS_score'].isin([1,2,3,4,5])]
print(min(df_heat['CS_score']))
print(max(df_heat['CS_score']))
df_heat.head()

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

# -------------------------------------------------------
# 1. AGGREGATE ARTICLE COUNTS BY COUNTRY (ADMIN0)
# -------------------------------------------------------
df_country = (
    df_final
        .explode("valid_SOURCEURL")
        .dropna(subset=["valid_SOURCEURL"])
        .groupby("ADMIN0")["valid_SOURCEURL"]
        .nunique()
        .reset_index()
)

df_country.rename(columns={"valid_SOURCEURL": "article_count"}, inplace=True)
df_country["ADMIN0"] = df_country["ADMIN0"].str.strip().str.title()

# -------------------------------------------------------
# 2. NATURAL EARTH NAME MAPPING (INCLUDES S. Sudan FIX)
# -------------------------------------------------------
ne_mapping = {
    "Burkina Faso": "Burkina Faso",
    "Burundi": "Burundi",
    "Cameroon": "Cameroon",
    "Central African Republic": "Central African Rep.",
    "Chad": "Chad",
    "Democratic Republic Of The Congo": "Dem. Rep. Congo",
    "Ethiopia": "Ethiopia",
    "Guinea": "Guinea",
    "Kenya": "Kenya",
    "Liberia": "Liberia",
    "Madagascar": "Madagascar",
    "Malawi": "Malawi",
    "Mali": "Mali",
    "Mauritania": "Mauritania",
    "Mozambique": "Mozambique",
    "Niger": "Niger",
    "Nigeria": "Nigeria",
    "Sierra Leone": "Sierra Leone",
    "Somalia": "Somalia",

    # ⭐ Correct name in Natural Earth
    "South Sudan": "S. Sudan",

    "Sudan": "Sudan",
    "Uganda": "Uganda",
    "Yemen": "Yemen",
    "Zambia": "Zambia",
    "Zimbabwe": "Zimbabwe",
}

df_country["NE_name"] = df_country["ADMIN0"].map(ne_mapping)

# -------------------------------------------------------
# 3. LOAD WORLD SHAPEFILE (Africa + Yemen)
# -------------------------------------------------------
world = gpd.read_file(gpd.datasets.get_path("naturalearth_lowres"))
world["name"] = world["name"].str.strip().str.title()

target = list(world[world["continent"] == "Africa"]["name"]) + ["Yemen"]
subset = world[world["name"].isin(target)].copy()

# -------------------------------------------------------
# 4. MERGE GEOMETRY WITH YOUR DATA
# -------------------------------------------------------
geo_df = subset.merge(df_country, how="left", left_on="name", right_on="NE_name")
geo_df["has_data"] = ~geo_df["article_count"].isna()

# -------------------------------------------------------
# 5. PLOT (GREY = NO DATA)
# -------------------------------------------------------
fig, ax = plt.subplots(figsize=(18, 16))

# Grey countries first
geo_df[geo_df["has_data"] == False].plot(
    color="lightgrey",
    edgecolor="black",
    linewidth=0.7,
    ax=ax
)

# Heatmap for countries with data
geo_df[geo_df["has_data"] == True].plot(
    column="article_count",
    cmap="YlOrRd",
    edgecolor="black",
    linewidth=0.7,
    legend=True,
    legend_kwds={"label": "Number of Articles", "shrink": 0.6},
    ax=ax
)

ax.set_title(
    "Geographic Heatmap of GDELT Events Article Counts by Country (Africa + Yemen)\nGrey = No Articles in Dataset",
    fontsize=18, fontweight="bold",
    pad=20
)

ax.axis("off")

# -------------------------------------------------------
# 6. LABELS
# -------------------------------------------------------
for idx, row in geo_df.iterrows():
    centroid = row.geometry.centroid
    ax.annotate(
        row["name"],
        xy=(centroid.x, centroid.y),
        ha="center",
        fontsize=9,
        color="black"
    )

plt.show()


In [ ]:
world = gpd.read_file(gpd.datasets.get_path("naturalearth_lowres"))

africa = world[world["continent"] == "Africa"]

print(sorted(africa["name"].unique()))


## Check correlation between articles and FS fluctuations

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf

# ======================================================
# 0. EXTRACT UNIQUE ENTITIES
# ======================================================
countries = df_final["ADMIN0"].unique()
periods = sorted(df_final["period"].astype(int).unique())

# Create full country–period panel
df_panel = pd.MultiIndex.from_product(
    [countries, periods], names=["ADMIN0", "period"]
).to_frame(index=False)

# ======================================================
# 1. ARTICLE COUNTS PER COUNTRY-PERIOD
# ======================================================
df_articles_period = (
    df_final
        .explode("valid_SOURCEURL")
        .dropna(subset=["valid_SOURCEURL"])
        .groupby(["ADMIN0", "period"])["valid_SOURCEURL"]
        .nunique()
        .reset_index(name="article_count")
)

# Ensure period is numeric
df_articles_period["period"] = df_articles_period["period"].astype(int)

# ======================================================
# 2. FOOD SECURITY PER COUNTRY-PERIOD
# ======================================================
df_fs_period = (
    df_final
        .groupby(["ADMIN0", "period"])["CS_score"]
        .max()
        .reset_index()
)

df_fs_period["period"] = df_fs_period["period"].astype(int)

df_fs_period_lag = df_fs_period.copy()
df_fs_period_lag = df_fs_period_lag[df_fs_period_lag['CS_score'].isin([1,2,3,4,5])]
df_fs_period_lag['prev_cs'] = df_fs_period_lag.groupby('ADMIN0')['CS_score'].shift(1)
df_fs_period_lag.drop('CS_score', axis=1, inplace=True)
# ======================================================
# 3. MERGE BOTH INTO THE COMPLETE PANEL
# ======================================================
df_period = (
    df_panel
        .merge(df_fs_period, on=["ADMIN0", "period"], how="left")
        .merge(df_articles_period, on=["ADMIN0", "period"], how="left")
        .merge(df_fs_period_lag, on=["ADMIN0", "period"], how="left")
)

def compute_articles_between(g):
    # g = one country's dataframe, already sorted by period
    fs_idx = g.index[g["CS_score"].notna()].tolist()   # indices of FEWSNET periods
    
    results = pd.Series(index=g.index, dtype="float")

    # For each pair of consecutive FEWSNET periods
    for i in range(1, len(fs_idx)):
        prev_i = fs_idx[i-1]
        curr_i = fs_idx[i]
        
        # sum article_count BETWEEN these periods (exclusive of prev, inclusive of curr)
        s = g.loc[prev_i:curr_i, "article_count"].sum() - g.loc[prev_i, "article_count"]

        # assign ONLY to the current FEWSNET period
        results[curr_i] = s

    return results

df_period["articles_between"] = (
    df_period
    .sort_values(["ADMIN0","period"])
    .groupby("ADMIN0", group_keys=False)
    .apply(compute_articles_between)
)


df_valid = df_period.copy()
df_valid = df_valid[df_valid["CS_score"].isin([1,2,3,4,5])]

# ======================================================
# 6. USE ACTUAL FOOD SECURITY LEVEL (CS_score)
# ======================================================
# Drop rows where article sums are missing or CS_score is missing
df_corr = df_valid.dropna(subset=["articles_between", "CS_score"])

# ======================================================
# 7. CORRELATION (raw, without controls)
# ======================================================
print("\nRaw correlation between CS_score and articles_between:")
print(df_corr[["CS_score", "articles_between"]].corr())

# ======================================================
# 8. SCATTERPLOT WITH REGRESSION LINE (raw)
# ======================================================
plt.figure(figsize=(10,7))
sns.regplot(
    data=df_corr,
    x="CS_score",
    y="articles_between",
    scatter_kws={"s": 60, "alpha": 0.7},
    line_kws={"color": "red"}
)
plt.title("Articles Between FEWSNET Periods vs Food Security Level (Raw Relationship)")
plt.xlabel("Food Security Level (CS_score)")
plt.ylabel("Articles Between Periods")
plt.tight_layout()
plt.show()

# ======================================================
# 9. OLS REGRESSION WITH FIXED EFFECTS
# ======================================================
# Country FE + Period FE
model_fe = smf.ols(
    formula="articles_between ~ CS_score + C(ADMIN0) + C(period)",
    data=df_corr
).fit()

print("\nOLS with Country & Period Fixed Effects (articles_between ~ CS_score + FE):")
print(model_fe.summary())

### Robustness: control for total country-month media volume

The regression above uses country and period fixed effects, which absorb time-invariant cross-country differences in coverage and global temporal trends. They do **not** absorb country-month idiosyncratic media-attention shocks that are unrelated to food security (e.g. elections, sports events) but which could correlate with our FS-keyword-filtered article counts.

To address this, we pull total country-month event volume from the public GDELT 2.0 BigQuery dataset (`gdelt-bq.gdeltv2.events`, no CAMEO / topic filter) and add it as a control. The `CS_score` coefficient in the controlled model isolates the within-country, within-period FS-specific signal, conditional on overall news activity in that country-month.


In [ ]:
# ======================================================
# 10. ROBUSTNESS: VOLUME + PRIOR-STATE CONTROLS, AND
#     HORSE-RACE BETWEEN FS-FILTERED AND TOTAL VOLUME
# ======================================================
# Two sets of specifications, all on the same sample:
#
#   Direction A: articles_between ~ CS_score (+ controls)
#       M1: original                 (CS_score + FE)
#       M2: + previous-period IPC    (CS_score + prev_cs + FE)
#       M3: + total media volume     (CS_score + total_articles_between + FE)
#       M4: + both controls          (CS_score + prev_cs + total_articles_between + FE)
#
#   Direction B: CS_score ~ count(s) (+ FE)   [predictive framing]
#       M5: FS count alone           (articles_between + FE)
#       M6: total count alone        (total_articles_between + FE)
#       M7: horse race               (articles_between + total_articles_between + FE)

fips_to_admin0 = {
    "UV": "burkina faso",  "BY": "burundi",        "CM": "cameroon",
    "CT": "central african republic",
    "CD": "chad",
    "CG": "democratic republic of the congo",
    "ET": "ethiopia",      "GV": "guinea",         "KE": "kenya",
    "LI": "liberia",       "MA": "madagascar",     "MI": "malawi",
    "ML": "mali",          "MR": "mauritania",     "MZ": "mozambique",
    "NG": "niger",         "NI": "nigeria",        "SL": "sierra leone",
    "SO": "somalia",       "OD": "south sudan",    "SU": "sudan",
    "UG": "uganda",        "YM": "yemen",          "ZA": "zambia",
    "ZI": "zimbabwe",
}

gdelt_total = pd.read_csv("../data/gdelt_counts.csv")
gdelt_total["ADMIN0"] = gdelt_total["country_code_fips"].map(fips_to_admin0)
gdelt_total = gdelt_total.dropna(subset=["ADMIN0"]).copy()
gdelt_total["period"] = gdelt_total["year_month"].str.replace("-", "").astype(int)

df_total = (
    gdelt_total[["ADMIN0", "period", "total_unique_articles"]]
    .rename(columns={"total_unique_articles": "total_article_count"})
)

df_period_ctrl = (
    df_period
    .merge(df_total, on=["ADMIN0", "period"], how="left")
    .fillna({"total_article_count": 0})
)

def compute_total_articles_between(g):
    fs_idx = g.index[g["CS_score"].notna()].tolist()
    out = pd.Series(index=g.index, dtype="float")
    for i in range(1, len(fs_idx)):
        prev_i, curr_i = fs_idx[i - 1], fs_idx[i]
        out[curr_i] = (
            g.loc[prev_i:curr_i, "total_article_count"].sum()
            - g.loc[prev_i, "total_article_count"]
        )
    return out

df_period_ctrl["total_articles_between"] = (
    df_period_ctrl
    .sort_values(["ADMIN0", "period"])
    .groupby("ADMIN0", group_keys=False)
    .apply(compute_total_articles_between)
)

df_corr_ctrl = (
    df_period_ctrl[df_period_ctrl["CS_score"].isin([1, 2, 3, 4, 5])]
    .dropna(subset=["articles_between", "CS_score",
                    "prev_cs", "total_articles_between"])
)

# ----- Direction A: explain article volume by severity -----
m1 = smf.ols("articles_between ~ CS_score + C(ADMIN0) + C(period)",
             data=df_corr_ctrl).fit()
m2 = smf.ols("articles_between ~ CS_score + prev_cs + C(ADMIN0) + C(period)",
             data=df_corr_ctrl).fit()
m3 = smf.ols("articles_between ~ CS_score + total_articles_between + C(ADMIN0) + C(period)",
             data=df_corr_ctrl).fit()
m4 = smf.ols("articles_between ~ CS_score + prev_cs + total_articles_between + C(ADMIN0) + C(period)",
             data=df_corr_ctrl).fit()

models_A = [("M1: baseline", m1),
            ("M2: + prev_cs", m2),
            ("M3: + volume", m3),
            ("M4: + both",   m4)]

def pretty(model, var):
    if var in model.params:
        return f"{model.params[var]:>9.3f} ({model.pvalues[var]:.3f})"
    return "      —      "

print("Direction A: articles_between ~ CS_score (+ controls) + FE")
print("=" * 84)
print(f"{'':22}" + "".join(f"{name:>16}" for name, _ in models_A))
print("-" * 84)
for var in ["CS_score", "prev_cs", "total_articles_between"]:
    row = f"{var:22}"
    for _, m in models_A:
        row += f"{pretty(m, var):>16}"
    print(row)
print("-" * 84)
print(f"{'R-squared':22}" + "".join(f"{m.rsquared:>16.3f}" for _, m in models_A))
print(f"{'N':22}" + "".join(f"{int(m.nobs):>16d}" for _, m in models_A))
print("=" * 84)

# ----- Direction B: predict severity from counts -----
m5 = smf.ols("CS_score ~ articles_between + C(ADMIN0) + C(period)",
             data=df_corr_ctrl).fit()
m6 = smf.ols("CS_score ~ total_articles_between + C(ADMIN0) + C(period)",
             data=df_corr_ctrl).fit()
m7 = smf.ols("CS_score ~ articles_between + total_articles_between + C(ADMIN0) + C(period)",
             data=df_corr_ctrl).fit()

models_B = [("M5: FS only",     m5),
            ("M6: total only",  m6),
            ("M7: horse race",  m7)]

print("\nDirection B: CS_score ~ count(s) + FE   [predictive framing]")
print("=" * 84)
print(f"{'':22}" + "".join(f"{name:>16}" for name, _ in models_B))
print("-" * 84)
for var in ["articles_between", "total_articles_between"]:
    row = f"{var:22}"
    for _, m in models_B:
        row += f"{pretty(m, var):>16}"
    print(row)
print("-" * 84)
print(f"{'R-squared':22}" + "".join(f"{m.rsquared:>16.3f}" for _, m in models_B))
print(f"{'Adj R-squared':22}" + "".join(f"{m.rsquared_adj:>16.3f}" for _, m in models_B))
print(f"{'N':22}" + "".join(f"{int(m.nobs):>16d}" for _, m in models_B))
print("=" * 84)
print("(values: coef (p-value); CS_score is ordinal so this is a linear approximation;"
      "\n high collinearity between the two counts inflates SEs in M7 — compare R^2)")

# Pearson correlation between the two counts (for context)
rho = df_corr_ctrl[["articles_between", "total_articles_between"]].corr().iloc[0, 1]
print(f"\nCorrelation between articles_between and total_articles_between: {rho:.3f}")

In [ ]:
# ======================================================
# 11. DOES FS-FILTERED NEWS RESPOND MORE THAN TOTAL NEWS?
# ======================================================
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Same panel as before, but also drop rows where total volume is 0
# (so the share/log are well-defined for ALL specs).
df_resp = (
    df_corr_ctrl
    .loc[df_corr_ctrl["total_articles_between"] > 0]
    .copy()
)
df_resp["fs_share"] = df_resp["articles_between"] / df_resp["total_articles_between"]
df_resp["log_fs"] = np.log1p(df_resp["articles_between"])
df_resp["log_total"] = np.log1p(df_resp["total_articles_between"])

# --- Level models ---
lvl_fs    = smf.ols("articles_between       ~ CS_score + C(ADMIN0) + C(period)", data=df_resp).fit()
lvl_total = smf.ols("total_articles_between ~ CS_score + C(ADMIN0) + C(period)", data=df_resp).fit()

# --- Log models (CS_score coef ≈ proportional change per IPC unit) ---
log_fs    = smf.ols("log_fs    ~ CS_score + C(ADMIN0) + C(period)", data=df_resp).fit()
log_total = smf.ols("log_total ~ CS_score + C(ADMIN0) + C(period)", data=df_resp).fit()

# --- Share model: does FS share itself rise with severity? ---
share     = smf.ols("fs_share  ~ CS_score + C(ADMIN0) + C(period)", data=df_resp).fit()

mean_fs    = df_resp["articles_between"].mean()
mean_total = df_resp["total_articles_between"].mean()
mean_share = df_resp["fs_share"].mean()

def line(label, model, dv_mean=None, pct=False):
    b = model.params["CS_score"]
    p = model.pvalues["CS_score"]
    extra = ""
    if dv_mean is not None:
        extra = f"   ({100*b/dv_mean:+.2f}% of mean DV per IPC unit)"
    if pct:
        extra = f"   ({100*b:+.2f}% per IPC unit)"
    print(f"{label:38} coef={b:>10.4f}  p={p:.4f}{extra}")

print("=" * 88)
print("Response of news volume to CS_score, country & period FE, same sample")
print(f"N = {int(lvl_fs.nobs)}")
print("-" * 88)
print("LEVEL MODELS (raw coefficient + relative effect):")
line("FS-filtered count    (level)", lvl_fs,    dv_mean=mean_fs)
line("Total count          (level)", lvl_total, dv_mean=mean_total)
print()
print("LOG MODELS (semi-elasticity — coef ≈ proportional change per IPC unit):")
line("FS-filtered count    (log)",   log_fs,    pct=True)
line("Total count          (log)",   log_total, pct=True)
print()
print("SHARE MODEL (does FS share itself rise with severity?):")
line(f"FS share (mean = {mean_share:.4f})", share)
print("=" * 88)
print(
    "Read:\n"
    "  - If FS-level coef/mean > Total-level coef/mean       -> FS filter amplifies (level)\n"
    "  - If log_fs CS_score > log_total CS_score             -> FS filter amplifies (proportional)\n"
    "  - If share CS_score > 0 and significant               -> FS share rises with severity"
)

# ======================================================
# PLOT: log(articles) vs CS_score, FS vs total
# (raw OLS line; FE-adjusted slope reported in annotation)
# ======================================================
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=False)

# --- Left panel: log(FS-filtered articles) vs CS_score ---
sns.regplot(
    data=df_resp,
    x="CS_score",
    y="log_fs",
    scatter_kws={"s": 55, "alpha": 0.6, "color": "#2166ac"},
    line_kws={"color": "red"},
    ax=axes[0],
)
axes[0].set_title("FS-filtered articles between assessments")
axes[0].set_xlabel("Food security level (CS score)")
axes[0].set_ylabel(r"$\log(1 + \mathrm{FS\ articles})$")

# --- Right panel: log(total articles) vs CS_score ---
sns.regplot(
    data=df_resp,
    x="CS_score",
    y="log_total",
    scatter_kws={"s": 55, "alpha": 0.6, "color": "#404040"},
    line_kws={"color": "red"},
    ax=axes[1],
)
axes[1].set_title("Total country–month articles (unfiltered)")
axes[1].set_xlabel("Food security level (CS score)")
axes[1].set_ylabel(r"$\log(1 + \mathrm{total\ articles})$")

# --- Annotate each panel with the FE-adjusted semi-elasticity ---
def fe_box(ax, model, label):
    b = model.params["CS_score"]
    p = model.pvalues["CS_score"]
    sig = "***" if p < 0.01 else ("**" if p < 0.05 else ("*" if p < 0.10 else ""))
    ax.text(
        0.03, 0.97,
        f"{label}\n"
        f"FE-adjusted slope: {b:+.3f}{sig}\n"
        f"({100*b:+.1f}% per IPC unit, p = {p:.4f})",
        transform=ax.transAxes,
        fontsize=10, va="top", ha="left",
        bbox=dict(boxstyle="round,pad=0.4", facecolor="white", edgecolor="0.7"),
    )

fe_box(axes[0], log_fs,    "FS-filtered (log)")
fe_box(axes[1], log_total, "Total (log)")

plt.tight_layout()
plt.savefig("../plots/log_articles_vs_cs.pdf", bbox_inches="tight", dpi=300)
plt.savefig("../plots/log_articles_vs_cs.png", bbox_inches="tight", dpi=300)
# Also drop into the LaTeX project directory:
plt.savefig("log_articles_vs_cs.png",
            bbox_inches="tight", dpi=300)

plt.show()

### Add lead lag analysis

In [ ]:
# ======================================================
# 11. LEAD-LAG DIAGNOSTIC: GDELT[t] -> IPC[t+k]
# ------------------------------------------------------
# For each country, build forward-shifted IPC targets cs_next_k
# (k = 1..4 future assessments) and regress them on contemporaneous
# FS-filtered article volume, with country and period fixed effects.
# Two specifications per horizon:
#   (a) cs_next_k ~ articles_between              + C(ADMIN0) + C(period)
#   (b) cs_next_k ~ articles_between + CS_score   + C(ADMIN0) + C(period)
# Spec (b) is the strong test: conditional on today's IPC, does
# today's GDELT still predict IPC at t+k? A non-trivial coefficient
# in (b) is direct evidence of forward-looking information beyond
# IPC persistence.
# ======================================================
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

HORIZONS = [1, 2, 3, 4]

df_lead = (
    df_period_ctrl[df_period_ctrl["CS_score"].isin([1, 2, 3, 4, 5])]
    .sort_values(["ADMIN0", "period"])
    .copy()
)
for k in HORIZONS:
    df_lead[f"cs_next_{k}"] = (
        df_lead.groupby("ADMIN0")["CS_score"].shift(-k)
    )

results = {}
for k in HORIZONS:
    target = f"cs_next_{k}"
    sample = (df_lead
              .dropna(subset=[target, "articles_between", "CS_score"])
              .copy())
    sample = sample[sample[target].isin([1, 2, 3, 4, 5])]
    if len(sample) < 30:
        results[k] = None
        continue
    fit_a = smf.ols(
        f"{target} ~ articles_between + C(ADMIN0) + C(period)",
        data=sample,
    ).fit()
    fit_b = smf.ols(
        f"{target} ~ articles_between + CS_score + C(ADMIN0) + C(period)",
        data=sample,
    ).fit()
    results[k] = {
        "n":         len(sample),
        "a_coef":    fit_a.params.get("articles_between", np.nan),
        "a_p":       fit_a.pvalues.get("articles_between", np.nan),
        "a_r2":      fit_a.rsquared,
        "b_coef":    fit_b.params.get("articles_between", np.nan),
        "b_p":       fit_b.pvalues.get("articles_between", np.nan),
        "b_cs_coef": fit_b.params.get("CS_score", np.nan),
        "b_cs_p":    fit_b.pvalues.get("CS_score", np.nan),
        "b_r2":      fit_b.rsquared,
    }

def _fmt(coef, pval):
    if coef is None or (isinstance(coef, float) and np.isnan(coef)):
        return "       —"
    star = "***" if pval < 0.01 else "**" if pval < 0.05 else "*" if pval < 0.10 else ""
    return f"{coef:>+8.3f} ({pval:.3f}){star}"

def _row(label, getter):
    cells = "".join(f"{getter(k):>20}" for k in HORIZONS)
    print(f"{label:<28}{cells}")

print("=" * 110)
print("LEAD-LAG DIAGNOSTIC: cs_next_k ~ articles_between (+ CS_score) + C(ADMIN0) + C(period)")
print("=" * 110)
print(f"{'Horizon (assessments)':<28}" + "".join(f"{'k='+str(k):>20}" for k in HORIZONS))
print("-" * 110)
print("(a) GDELT only:")
_row(
    "    articles_between",
    lambda k: _fmt(results[k]["a_coef"], results[k]["a_p"]) if results[k] else "       —",
)
_row(
    "    R^2",
    lambda k: f"{results[k]['a_r2']:.3f}" if results[k] else "—",
)
print()
print("(b) Conditional on today's IPC (CS_score):")
_row(
    "    articles_between",
    lambda k: _fmt(results[k]["b_coef"], results[k]["b_p"]) if results[k] else "       —",
)
_row(
    "    CS_score",
    lambda k: _fmt(results[k]["b_cs_coef"], results[k]["b_cs_p"]) if results[k] else "       —",
)
_row(
    "    R^2",
    lambda k: f"{results[k]['b_r2']:.3f}" if results[k] else "—",
)
_row(
    "    N",
    lambda k: f"{results[k]['n']:d}" if results[k] else "—",
)
print("=" * 110)
print("coef (p-value); ***p<0.01, **p<0.05, *p<0.10")
print()
print("(a) Does GDELT at time t predict IPC at t+k unconditionally?")
print("(b) Conditional on today's IPC, does GDELT at t still predict IPC at t+k?")
print("    A non-zero, signed coefficient in (b) -> forward-looking signal beyond IPC persistence.")

### ADMIN1 distribution

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# --- Compute unique article counts per ADMIN1 ---
admin1_counts = (
    df_final
        .explode("valid_SOURCEURL")
        .groupby("ADMIN1")["valid_SOURCEURL"]
        .nunique()
        .reset_index(name="article_count")
)

# --- FIX: ensure article_count is integer ---
admin1_counts["article_count"] = admin1_counts["article_count"].astype(int)

# --- Correct bucket definitions ---
bins   = [-1, 0, 5, 10, 20, 50, float("inf")]
labels = ["0", "1–5", "6–10", "11–20", "21–50", "50+"]

admin1_counts["bucket"] = pd.cut(
    admin1_counts["article_count"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

# --- Correct distribution: count ADMIN1 per bucket ---
distribution = (
    admin1_counts["bucket"]
    .value_counts()
    .reindex(labels)
    .reset_index()
)

distribution.columns = ["bucket", "num_admin1"]

# --- Plot ---
plt.figure(figsize=(12,6))
sns.barplot(data=distribution, x="bucket", y="num_admin1", color="steelblue")
plt.title("ADMIN1 Distribution by Article Count Buckets (GDELT Events)")
plt.xlabel("Article Count Bucket")
plt.ylabel("Number of ADMIN1 Regions")
plt.tight_layout()
plt.show()


## Admin2 check

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# --- Compute unique article counts per ADMIN1 ---
admin2_counts = (
    df_final
        .explode("valid_SOURCEURL")
        .groupby("ADMIN2")["valid_SOURCEURL"]
        .nunique()
        .reset_index(name="article_count")
)

# --- FIX: ensure article_count is integer ---
admin2_counts["article_count"] = admin2_counts["article_count"].astype(int)

# --- Correct bucket definitions ---
bins   = [-1, 0, 5, 10, 20, 50, float("inf")]
labels = ["0", "1–5", "6–10", "11–20", "21–50", "50+"]

admin2_counts["bucket"] = pd.cut(
    admin2_counts["article_count"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

# --- Correct distribution: count ADMIN1 per bucket ---
distribution = (
    admin2_counts["bucket"]
    .value_counts()
    .reindex(labels)
    .reset_index()
)

distribution.columns = ["bucket", "num_admin1"]

# --- Plot ---
plt.figure(figsize=(12,6))
sns.barplot(data=distribution, x="bucket", y="num_admin1", color="steelblue")
plt.title("ADMIN2 Distribution by Article Count Buckets (GDELT Events)")
plt.xlabel("Article Count Bucket")
plt.ylabel("Number of ADMIN2 districts")
plt.tight_layout()
plt.show()


## Heatmap for each topic metric

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# ---------------------------------------------------------
# 0. CHOOSE OUTPUT FOLDER
# ---------------------------------------------------------
output_folder = "/Users/marco.bertetti/Desktop/charts/nlp/events"
os.makedirs(output_folder, exist_ok=True)

# ---------------------------------------------------------
# 1. LIST OF ALL METRICS YOU WANT TO INCLUDE
# ---------------------------------------------------------
metric_list_columns = [
    'fatalities_freq_list', 'fatalities_count_list', 
    'displaced_freq_list', 'displaced_count_list',
    'detained_freq_list', 'detained_count_list',
    'injured_freq_list', 'injured_count_list',
    'sexual_violence_freq_list', 'sexual_violence_count_list',
    'torture_freq_list', 'torture_count_list',
    'economic_shocks_freq_list', 'agriculture_freq_list',
    'weather_freq_list', 'food_insecurity_freq_list',
    'pred_resource_food_list', 'pred_resource_water_list', 'pred_resource_cash_aid_list',
    'pred_resource_healthcare_list', 'pred_resource_shelter_list',
    'pred_resource_livelihoods_list', 'pred_resource_education_list',
    'pred_resource_infrastructure_list',
    'NumMentions', 'NumSources', 'NumArticles'
]

# ---------------------------------------------------------
# 2. SAFE SUM FUNCTION
# ---------------------------------------------------------
def safe_sum(x):
    if isinstance(x, list):
        clean = []
        for v in x:
            try:
                clean.append(float(v))
            except:
                clean.append(0)
        return sum(clean)
    return 0

# ---------------------------------------------------------
# 3. APPLY SAFE SUM TO ALL LIST COLUMNS
# ---------------------------------------------------------
for col in metric_list_columns:
    new_col = col.replace("_list", "")
    df_final[new_col] = df_final[col].apply(safe_sum)

all_metrics = [c.replace("_list", "") for c in metric_list_columns]

print("Total metrics:", len(all_metrics))
print(all_metrics)

# ---------------------------------------------------------
# 4. GENERATE HEATMAP FOR EACH METRIC (SAVE FIRST, THEN SHOW)
# ---------------------------------------------------------
for metric in all_metrics:

    df_heat = (
        df_final
            .groupby(["ADMIN0", "period"])[metric]
            .sum()
            .unstack(fill_value=0)
            .sort_index()
            .sort_index(axis=1)
    )

    df_heat.index = df_heat.index.str.title()

    plt.figure(figsize=(35, 16))
    ax = sns.heatmap(
        df_heat,
        cmap="YlOrRd",
        linewidths=0.2,
        linecolor="gray",
        annot=True,
        fmt="g",
        annot_kws={"fontsize": 7, "color": "black"},
        cbar_kws={"shrink": 0.5}
    )

    ax.set_title(f"{metric.upper()} (GDELT Events) by Country and Period", fontsize=26, fontweight='bold', pad=20)
    ax.set_xlabel("Period (YYYYMM)", fontsize=20)
    ax.set_ylabel("Country (ADMIN0)", fontsize=20)

    ax.set_xticklabels(df_heat.columns, fontsize=10, rotation=90)
    ax.set_yticklabels(df_heat.index, fontsize=14)

    plt.tight_layout()

    # ---- SAVE BEFORE SHOW ----
    file_path = os.path.join(output_folder, f"{metric.upper()}.png")
    plt.savefig(file_path, dpi=300)
    print(f"Saved: {file_path}")

    # ---- NOW SHOW ----
    plt.show()

    plt.close()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import os

# ---------------------------------------------------------
# 0. CHOOSE OUTPUT FOLDER
# ---------------------------------------------------------
output_folder = "/Users/marco.bertetti/Desktop/charts/nlp/events/impact_urgency"
os.makedirs(output_folder, exist_ok=True)

# ---------------------------------------------------------
# 1. Convert pred_impact_type_list into category frequency columns
# ---------------------------------------------------------
def count_categories(list_value):
    if isinstance(list_value, list):
        return Counter(list_value)
    return Counter()

# Impact type categories → expand to columns
impact_counts = df_final["pred_impact_type_list"].apply(count_categories)
impact_df = pd.DataFrame(impact_counts.tolist()).fillna(0)
impact_df.columns = [f"impact_{c}" for c in impact_df.columns]

# ---------------------------------------------------------
# 2. Convert pred_urgency_list into category frequency columns
# ---------------------------------------------------------
urgency_counts = df_final["pred_urgency_list"].apply(count_categories)
urgency_df = pd.DataFrame(urgency_counts.tolist()).fillna(0)
urgency_df.columns = [f"urgency_{c}" for c in urgency_df.columns]

# ---------------------------------------------------------
# 3. Merge back into df_final
# ---------------------------------------------------------
df_final2 = pd.concat([df_final, impact_df, urgency_df], axis=1)

# All expanded metric names
categorical_metrics = list(impact_df.columns) + list(urgency_df.columns)

print("Extracted categorical metrics:")
print(categorical_metrics)

# ---------------------------------------------------------
# 4. Generate heatmaps for each categorical metric
# ---------------------------------------------------------
for metric in categorical_metrics:

    df_heat = (
        df_final2
            .groupby(["ADMIN0", "period"])[metric]
            .sum()
            .unstack(fill_value=0)
            .sort_index()
            .sort_index(axis=1)
    )

    # Capitalize country names
    df_heat.index = df_heat.index.str.title()

    # ---- PLOT ----
    plt.figure(figsize=(35, 16))
    ax = sns.heatmap(
        df_heat,
        cmap="Blues",
        linewidths=0.2,
        linecolor="gray",
        annot=True,                      # SHOW NUMBERS
        fmt="g",
        annot_kws={"fontsize": 7, "color": "black"},   # ALWAYS BLACK TEXT
        cbar_kws={"shrink": 0.5}
    )

    ax.set_title(f"{metric.upper()} by Country and Period",
                 fontsize=26, fontweight='bold', pad=20)
    ax.set_xlabel("Period (YYYYMM)", fontsize=20)
    ax.set_ylabel("Country (ADMIN0)", fontsize=20)

    ax.set_xticklabels(df_heat.columns, fontsize=10, rotation=90)
    ax.set_yticklabels(df_heat.index, fontsize=14)

    plt.tight_layout()

    # ---- SAVE BEFORE SHOW ----
    file_path = os.path.join(output_folder, f"{metric.upper()}.png")
    plt.savefig(file_path, dpi=300)
    print(f"Saved: {file_path}")

    # ---- SHOW IN NOTEBOOK ----
    plt.show()

    plt.close()

Highly reliable

conflict

death

displacement

weather shock

urgency

Useful but redundant

injury

frequency_count

death_count

Noisy / probably low predictive power

economic loss

policy change

disease (depending on IPC label relationship)